In [3]:
import nltk
import keras
import pandas as pd
from clean_text import clean_text
from keras.models import Sequential
from nltk.tokenize import sent_tokenize
from keras.callbacks import EarlyStopping
from keras.layers.embeddings import Embedding
from keras.preprocessing.text import Tokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Dense, LSTM, SpatialDropout1D

In [4]:
train_df = pd.read_csv("train.csv")
copy_train_df = train_df.copy()
copy_train_df.drop(["id", "discourse_id", "discourse_start", "discourse_end", "discourse_type_num", "predictionstring"], axis=1, inplace=True)

In [5]:
discourse_text = []
discourse_type = []
for i,paragraph in enumerate(copy_train_df.discourse_text):
    types = []
    phrases = sent_tokenize(paragraph)
    types.append(copy_train_df.discourse_type[i])
    discourse_text.extend(phrases)
    discourse_type.extend(types*len(phrases))

In [6]:
copy_train_df = pd.DataFrame({"discourse_text":discourse_text, "discourse_type":discourse_type})

In [7]:
copy_train_df.discourse_text = copy_train_df.discourse_text.apply(clean_text)

In [8]:
tk = Tokenizer(num_words=None, filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
tk.fit_on_texts(copy_train_df.discourse_text)

In [9]:
print(list(tk.word_counts.items())[:10])

[('modern', 469), ('human', 6225), ('today', 2335), ('always', 5731), ('phone', 15134), ('nbr', 26104), ('hour', 2094), ('day', 8685), ('stop', 2134), ('text', 2973)]


In [10]:
dico_size = len(list(tk.word_counts.items()))
print(dico_size)

49401


In [11]:
print(sorted(list(tk.word_counts.items()), key= lambda x: -x[1])[:10])

[('student', 68421), ('would', 45303), ('people', 43925), ('school', 37504), ('car', 35353), ('nbr', 26104), ('get', 25329), ('help', 24188), ('one', 24021), ('could', 23829)]


In [12]:
print(list(tk.word_index.items())[:10])

[('student', 1), ('would', 2), ('people', 3), ('school', 4), ('car', 5), ('nbr', 6), ('get', 7), ('help', 8), ('one', 9), ('could', 10)]


In [13]:
seqTrain = tk.texts_to_sequences(copy_train_df.discourse_text)
print(seqTrain[:3])

[[925, 84, 245, 93, 23], [93, 23, 6, 272, 57, 269, 191, 159, 1042, 337, 2110, 260, 778], [29, 41]]


In [14]:
print(copy_train_df.discourse_text[:1])

0    modern human today always phone
Name: discourse_text, dtype: object


In [15]:
print(list(tk.word_index.items())[933])

('fear', 934)


In [16]:
max_length = max([len(doc) for doc in seqTrain])
print(max_length)

346


In [17]:
padTrain = pad_sequences(seqTrain, maxlen=max_length, padding='post')

In [18]:
print(padTrain[:3])

[[925  84 245 ...   0   0   0]
 [ 93  23   6 ...   0   0   0]
 [ 29  41   0 ...   0   0   0]]


In [19]:
Y = pd.get_dummies(copy_train_df.discourse_type).values
print('Shape of label tensor:', Y.shape)

Shape of label tensor: (331224, 7)


In [20]:
X_train, X_test, y_train, y_test = train_test_split(padTrain, Y, test_size=0.2, random_state=123)

In [21]:
print('x_train shape:', X_train.shape)
print('x_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

x_train shape: (264979, 346)
x_test shape: (66245, 346)
y_train shape: (264979, 7)
y_test shape: (66245, 7)


In [22]:
output_dim = len(copy_train_df.discourse_type.unique())

In [25]:
pmc = Sequential()
pmc.add(Embedding(input_dim=dico_size+1, output_dim=10, input_length=max_length))
pmc.add(SpatialDropout1D(0.2))
pmc.add(LSTM(max_length, dropout=0.2, recurrent_dropout=0.2))
pmc.add(Dense(7, activation='softmax'))
pmc.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print(pmc.summary())

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 346, 10)           494020    
                                                                 
 spatial_dropout1d_1 (Spatia  (None, 346, 10)          0         
 lDropout1D)                                                     
                                                                 
 lstm_1 (LSTM)               (None, 346)               494088    
                                                                 
 dense_1 (Dense)             (None, 7)                 2429      
                                                                 
Total params: 990,537
Trainable params: 990,537
Non-trainable params: 0
_________________________________________________________________
None


In [26]:
history = pmc.fit(X_train, y_train, epochs=2, batch_size=64, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss', patience=3, min_delta=0.0001)])

Epoch 1/2
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
2859/3313 [========================>.....] - ETA: 1:02:22 - loss: 1.4419 - accuracy: 0.5207

KeyboardInterrupt: 